# Clasificador de Emoción Musical — CNN + BiLSTM + Self-Attention (versión Kaggle)

Adaptación para **Kaggle Notebooks** del proyecto original (Google Colab). Antes de ejecutar:

1. Sube el dataset **DEAM** (audio + anotaciones) como un *Kaggle Dataset* y añádelo a este notebook con **Add Data**. Puede subirse ya descomprimido (recomendado) o como los `.zip` originales (`DEAM_audio.zip`, `DEAM_Annotations.zip`, `DEAM_Features.zip`).
2. Ajusta la variable `DATASET_SLUG` de la siguiente celda con el nombre de la carpeta que aparece bajo `/kaggle/input/` para tu dataset.
3. Activa un acelerador **GPU** en Settings del notebook para acelerar el entrenamiento.

A diferencia de Colab, `/kaggle/input` es de solo lectura y `/kaggle/working` es el único directorio con permisos de escritura (además, su contenido no persiste entre sesiones salvo que lo guardes como *Output* del notebook).

## Paso 1: Localizar el Dataset en Kaggle

In [ ]:
import os

# Nombre de la carpeta del dataset dentro de /kaggle/input (ajústalo al tuyo)
DATASET_SLUG = 'deam-mediaeval-dataset-emotional-analysis-in-music'

input_dir = os.path.join('/kaggle/input', DATASET_SLUG)
work_dir = '/kaggle/working'
os.chdir(work_dir)

print(f"Directorio de entrada (solo lectura): {input_dir}")
print(f"Directorio de trabajo (lectura/escritura): {os.getcwd()}")

if not os.path.isdir(input_dir):
    print(f"\n⚠️ No se encontró {input_dir}.")
    print("Revisa el panel derecho 'Input' del notebook y actualiza DATASET_SLUG con el nombre exacto de la carpeta.")
else:
    print(f"\nContenido encontrado: {os.listdir(input_dir)}")

## Paso 2: Preparar carpetas DEAM_audio / DEAM_Annotations / DEAM_Features

Si el dataset se subió ya descomprimido, usamos las carpetas directamente desde `/kaggle/input` (sin copiarlas, para no duplicar espacio). Si en cambio se subieron los `.zip`, los extraemos una sola vez a `/kaggle/working`.

In [ ]:
import zipfile
import os

zip_files = ['DEAM_audio.zip', 'DEAM_Annotations.zip', 'DEAM_Features.zip']
folder_names = ['DEAM_audio', 'DEAM_Annotations', 'DEAM_Features']

# Rutas efectivas que usará el resto del notebook para cada carpeta
deam_paths = {}

for zip_file, folder in zip(zip_files, folder_names):
    input_folder = os.path.join(input_dir, folder)
    input_zip = os.path.join(input_dir, zip_file)
    local_folder = os.path.join(work_dir, folder)

    if os.path.isdir(input_folder):
        # Ya viene descomprimido en el dataset de Kaggle: se usa directamente
        deam_paths[folder] = input_folder
        print(f'{folder}: usando carpeta ya extraída en el dataset -> {input_folder}')
    elif os.path.isdir(local_folder) and os.listdir(local_folder):
        deam_paths[folder] = local_folder
        print(f'{folder}: ya extraída previamente en esta sesión -> {local_folder}')
    elif os.path.exists(input_zip):
        print(f'Extrayendo {input_zip} a {local_folder}...')
        os.makedirs(local_folder, exist_ok=True)
        try:
            with zipfile.ZipFile(input_zip, 'r') as zip_ref:
                zip_ref.extractall(local_folder)
            deam_paths[folder] = local_folder
            print(f'¡Éxito! Contenido extraído en: {local_folder}')
        except zipfile.BadZipFile:
            print(f'Error: {input_zip} no es un archivo ZIP válido.')
    else:
        print(f'⚠️ No se encontró ni {input_folder} ni {input_zip}. Revisa el dataset añadido.')

print(f"\nRutas finales: {deam_paths}")

## Paso 3: Cargar Anotaciones y Preparar LabelEncoder

Necesitamos cargar las anotaciones de las emociones para cada canción y preparar un `LabelEncoder` para convertir las etiquetas de texto (Alegre, Neutra, Triste) a valores numéricos, que es lo que el modelo de machine learning espera.

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import os

annotations_dir = deam_paths['DEAM_Annotations']
csv_files = []

for root, dirs, files in os.walk(annotations_dir):
    for file in files:
        if file.endswith('.csv') and 'averaged_songs' in file.lower() and 'averaged' in root.lower():
            csv_files.append(os.path.join(root, file))

if not csv_files:
    for root, dirs, files in os.walk(annotations_dir):
        for file in files:
            if file.endswith('.csv') and 'averaged' in file.lower():
                csv_files.append(os.path.join(root, file))

print(f"Archivos de anotaciones promedio encontrados: {csv_files}")

df_list = []
for file_path in csv_files:
    df_temp = pd.read_csv(file_path)
    df_temp.columns = df_temp.columns.str.strip()
    df_list.append(df_temp)

df_annotations = pd.concat(df_list, ignore_index=True)
print(f"Total de canciones únicas cargadas: {len(df_annotations)}")

arousal_name = [col for col in df_annotations.columns if 'arousal' in col.lower() and 'mean' in col.lower()][0]
valence_name = [col for col in df_annotations.columns if 'valence' in col.lower() and 'mean' in col.lower()][0]
song_id_name = [col for col in df_annotations.columns if 'song_id' in col.lower() or 'id' in col.lower()][0]

df_avg = df_annotations[[song_id_name, arousal_name, valence_name]].copy()
df_avg.rename(columns={song_id_name: 'song_id', arousal_name: 'Arousal', valence_name: 'Valence'}, inplace=True)

df_avg.dropna(subset=['Arousal', 'Valence'], inplace=True)

def classify_emotion(row):
    if row['Arousal'] > 5 and row['Valence'] > 5:
        return 'Alegre'
    elif row['Arousal'] <= 5 and row['Valence'] <= 5:
        return 'Triste'
    else:
        return 'Neutra'

df_avg['Emocion'] = df_avg.apply(classify_emotion, axis=1)

le = LabelEncoder()
df_avg['emotion_encoded'] = le.fit_transform(df_avg['Emocion'])

print("\ndf_avg (primeras 5 filas limpias):")
display(df_avg.head())
print(f"Distribución de clases:\n{df_avg['Emocion'].value_counts()}")
print("\n¡LabelEncoder (le) y df_avg están listos de forma correcta y sin valores nulos!")

## Paso 4: Extracción de Características a Nivel de Frame (CNN + LSTM)

Extraemos las características directamente del **audio crudo** de DEAM, con la misma llamada de `opensmile` que usará la inferencia (Paso 8), para evitar el desajuste de dominio entre entrenamiento e inferencia.

In [ ]:
!pip install -q opensmile

In [ ]:
import pandas as pd
import numpy as np
import opensmile
from tqdm.notebook import tqdm
import os

if 'le' not in locals():
    raise NameError("El LabelEncoder (le) no está definido. Ejecuta primero el Paso 3.")

max_sequence_length = 0
all_sequences = []
all_labels = []

audio_dir = os.path.join(deam_paths['DEAM_audio'], 'MEMD_audio')
if not os.path.isdir(audio_dir):
    # Algunos uploads de Kaggle aplanan la estructura; probamos la carpeta raíz como fallback
    audio_dir = deam_paths['DEAM_audio']

# Cache en /kaggle/working: se pierde al terminar la sesión salvo que se
# guarde como Output del notebook (Save Version -> Save & Run All).
cache_dir = os.path.join(work_dir, 'DEAM_LLD_cache')
os.makedirs(cache_dir, exist_ok=True)

smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.ComParE_2016,
    feature_level=opensmile.FeatureLevel.LowLevelDescriptors,
)

print("Extrayendo características LLD (ComParE_2016) desde el audio crudo de DEAM...")
print(f"Caché en: {cache_dir}")

df_avg_sorted = df_avg.sort_values(by='song_id').reset_index(drop=True)

for idx, row in tqdm(df_avg_sorted.iterrows(), total=len(df_avg_sorted), desc="Extrayendo características"):
    song_id = int(row['song_id'])
    emotion = row['Emocion']

    cache_path = os.path.join(cache_dir, f'{song_id}.npy')

    try:
        if os.path.exists(cache_path):
            sequence = np.load(cache_path)
        else:
            audio_path = os.path.join(audio_dir, f'{song_id}.mp3')
            if not os.path.exists(audio_path):
                continue
            feats = smile.process_file(audio_path)
            sequence = feats.select_dtypes(include=[np.number]).values.astype('float32')
            np.save(cache_path, sequence)

        if sequence.shape[1] > 0:
            all_sequences.append(sequence)
            all_labels.append(le.transform([emotion])[0])

            if sequence.shape[0] > max_sequence_length:
                max_sequence_length = sequence.shape[0]

    except Exception as e:
        continue

print(f"\nCaracterísticas temporales cargadas correctamente para {len(all_sequences)} canciones.")
print(f"Longitud máxima de secuencia encontrada: {max_sequence_length} frames.")
print(f"Número de características por frame: {all_sequences[0].shape[1] if all_sequences else 0}")

X_cnn_lstm = all_sequences
y_cnn_lstm = np.array(all_labels)

## Paso 4.1: Preprocesamiento de Datos (Padding y Split)

In [ ]:
import tensorflow as tf
import keras
try:
    from keras.utils import pad_sequences
except ImportError:
    from keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split
import numpy as np

if 'X_cnn_lstm' not in locals() or not X_cnn_lstm:
    raise ValueError("No se encontraron secuencias de características en 'X_cnn_lstm'. Vuelve a ejecutar el Paso 4.")

num_features_per_frame = X_cnn_lstm[0].shape[1]

lengths = np.array([seq.shape[0] for seq in X_cnn_lstm])
print(f"Longitud de secuencia — min: {lengths.min()}, mediana: {int(np.median(lengths))}, "
      f"p95: {int(np.percentile(lengths, 95))}, máxima: {lengths.max()} frames.")

# Usar la longitud máxima real dispararía un array de decenas de GB, que
# revienta la RAM disponible en Kaggle. Usamos el percentil 95 como tope.
max_seq_len = int(np.percentile(lengths, 95))
print(f"Número de características por frame: {num_features_per_frame}")
print(f"Realizando padding/truncado a {max_seq_len} frames (percentil 95)...")

estimated_gb = len(X_cnn_lstm) * max_seq_len * num_features_per_frame * 4 / (1024 ** 3)
print(f"Tamaño estimado de X_padded: ~{estimated_gb:.2f} GB")

X_padded = pad_sequences(X_cnn_lstm, maxlen=max_seq_len, dtype='float32', padding='post', truncating='post', value=0.0)

print(f"Forma final de los datos transformados (X_padded): {X_padded.shape}")

X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(
    X_padded, y_cnn_lstm, test_size=0.2, random_state=42, stratify=y_cnn_lstm
)

print(f"\nTamaño de X_train para CNN+LSTM: {X_train_cnn.shape}")
print(f"Tamaño de X_test para CNN+LSTM: {X_test_cnn.shape}")
print(f"Tamaño de y_train para CNN+LSTM: {y_train_cnn.shape}")
print(f"Tamaño de y_test para CNN+LSTM: {y_test_cnn.shape}")

## Paso 4.2: Balanceo del Conjunto de Entrenamiento

Para evitar que nuestro modelo se sesgue hacia la clase mayoritaria, aplicaremos un submuestreo en el conjunto de entrenamiento de manera que todas las emociones tengan el mismo número de muestras (igualando a la clase minoritaria).

In [ ]:
import numpy as np
from collections import Counter

print("Distribución original de y_train_cnn:", Counter(y_train_cnn))

class_counts = Counter(y_train_cnn)
min_class_size = min(class_counts.values())
print(f"Tamaño de la clase minoritaria en entrenamiento: {min_class_size}")

balanced_indices = []

for emotion_class in class_counts.keys():
    indices = np.where(y_train_cnn == emotion_class)[0]
    chosen_indices = np.random.choice(indices, size=min_class_size, replace=False)
    balanced_indices.extend(chosen_indices)

balanced_indices = np.array(balanced_indices)
np.random.shuffle(balanced_indices)

X_train_cnn_balanced = X_train_cnn[balanced_indices]
y_train_cnn_balanced = y_train_cnn[balanced_indices]

print("\n--- Dataset Balanceado con Éxito ---")
print(f"Distribución nueva de y_train_cnn_balanced: {Counter(y_train_cnn_balanced)}")
print(f"Nueva forma de X_train_cnn_balanced: {X_train_cnn_balanced.shape}")

## Paso 5: Implementación de Fuzzy Classification (Soft Labels)

En lugar de clasificaciones estrictas (Hard Labels), vamos a calcular **Soft Labels**.
1. Encontramos el **centroide** (media de Arousal y Valence) para cada emoción.
2. Calculamos la **distancia euclidiana** de cada canción a los tres centroides.
3. Aplicamos una función **softmax sobre las distancias negativas** para convertirlas en probabilidades.

In [ ]:
import numpy as np
from scipy.spatial import distance
import pandas as pd

centros_emocionales = df_avg.groupby('Emocion')[['Valence', 'Arousal']].mean()
print("--- Centros Emocionales Calculados ---")
display(centros_emocionales)

def calcular_soft_labels(row, centros, beta=1.5):
    punto = np.array([row['Valence'], row['Arousal']])
    distancias = []
    emociones = centros.index.tolist()

    for emocion in emociones:
        centro = centros.loc[emocion].values
        dist = distance.euclidean(punto, centro)
        distancias.append(dist)

    distancias = np.array(distancias)
    exp_dists = np.exp(-beta * distancias)
    probabilidades = exp_dists / np.sum(exp_dists)

    return pd.Series({f'prob_{emocion}': prob for emocion, prob in zip(emociones, probabilidades)})

soft_labels_df = df_avg_sorted.apply(lambda row: calcular_soft_labels(row, centros_emocionales, beta=1.5), axis=1)

df_soft = pd.concat([df_avg_sorted, soft_labels_df], axis=1)

print("\n--- Muestra de Canciones con sus Soft Labels ---")
display(df_soft.sample(5, random_state=42))

In [ ]:
from sklearn.model_selection import train_test_split

columnas_prob = [f'prob_{clase}' for clase in le.classes_]
y_soft_all = df_soft[columnas_prob].values

print(f"Las clases del modelo esperan este orden: {le.classes_}")
print(f"Forma de y_soft_all: {y_soft_all.shape}")
print(f"Ejemplo del primer elemento de y_soft_all:\n{y_soft_all[0]} (Suma: {np.sum(y_soft_all[0]):.2f})\n")

X_train_cnn, X_test_cnn, y_train_soft, y_test_soft = train_test_split(
    X_padded, y_soft_all, test_size=0.2, random_state=42, stratify=y_cnn_lstm
)

print(f"Tamaño de X_train_cnn: {X_train_cnn.shape}")
print(f"Tamaño de y_train_soft: {y_train_soft.shape}")
print(f"Tamaño de X_test_cnn: {X_test_cnn.shape}")
print(f"Tamaño de y_test_soft: {y_test_soft.shape}")

print("\n¡Los conjuntos de datos con Soft Labels están listos para entrenar!")

In [ ]:
import keras
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

if 'X_train_cnn_balanced' not in locals() or 'y_train_soft' not in locals() or 'balanced_indices' not in locals():
    raise NameError("Faltan variables necesarias. Asegúrate de ejecutar las celdas de balanceo y de creación de Soft Labels.")

X_train_to_use = X_train_cnn_balanced
y_train_to_use = y_train_soft[balanced_indices]

max_seq_len = X_train_to_use.shape[1]
num_features_per_frame = X_train_to_use.shape[2]
num_classes = len(le.classes_)

print(f"Dataset balanceado y con Soft Labels listo para el modelo con Atención.")
print(f"Forma de entrada: ({max_seq_len}, {num_features_per_frame})")
print(f"Número de clases a predecir: {num_classes} ({le.classes_})")

early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.00001)

## Paso 6: Clasificación con Mecanismo de Atención (Self-Attention)

Entrenamos nuestro modelo de clasificación definitivo usando el dataset balanceado y las *Soft Labels*, con una capa de **Self-Attention** que permite al modelo enfocarse en los momentos clave de la canción.

In [ ]:
import tensorflow as tf
from keras.models import Model
from keras.layers import Input, Conv1D, MaxPooling1D, LSTM, Dense, Dropout, Bidirectional, BatchNormalization, Attention, GlobalAveragePooling1D, Concatenate

print("Definiendo modelo CNN + BiLSTM + Self-Attention para CLASIFICACIÓN.")

inputs = Input(shape=(max_seq_len, num_features_per_frame))

x = Conv1D(filters=128, kernel_size=5, activation='relu', padding='same')(inputs)
x = BatchNormalization()(x)
x = MaxPooling1D(pool_size=8)(x)
x = Dropout(0.3)(x)

x = Bidirectional(LSTM(128, return_sequences=True))(x)
x = Dropout(0.3)(x)

attention_out = Attention()([x, x])

x = Concatenate()([x, attention_out])

x = GlobalAveragePooling1D()(x)

x = Dense(64, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

outputs = Dense(num_classes, activation='softmax')(x)

model_attention = Model(inputs=inputs, outputs=outputs)

model_attention.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_attention.summary()

print("\nEntrenando el modelo con Atención...")
history_att = model_attention.fit(
    X_train_to_use, y_train_to_use,
    epochs=40,
    batch_size=32,
    validation_data=(X_test_cnn, y_test_soft),
    callbacks=[early_stopping, reduce_lr]
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_att.history['accuracy'], label='Accuracy Train')
plt.plot(history_att.history['val_accuracy'], label='Accuracy Val')
plt.title('Precisión - Modelo con Atención')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_att.history['loss'], label='Loss Train')
plt.plot(history_att.history['val_loss'], label='Loss Val')
plt.title('Pérdida - Modelo con Atención')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print("\n--- Evaluando el Modelo con Atención en el Set de Prueba ---")
y_pred_att_probs = model_attention.predict(X_test_cnn)
y_pred_att_discrete = np.argmax(y_pred_att_probs, axis=1)
y_test_true_discrete = np.argmax(y_test_soft, axis=1)

print("\nReporte de Clasificación:")
print(classification_report(y_test_true_discrete, y_pred_att_discrete, target_names=le.classes_))

cm_att = confusion_matrix(y_test_true_discrete, y_pred_att_discrete)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_att, annot=True, fmt='d', cmap='Greens', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Matriz de Confusión: Modelo con Atención')
plt.xlabel('Predicción')
plt.ylabel('Verdadera')
plt.show()

## Paso 7: Probar el Modelo con un Archivo de Audio Propio

Kaggle no tiene un selector de archivos nativo tipo `tkinter` (no hay entorno gráfico). Para probar el modelo con tu propio audio:

1. Sube el `.wav`/`.mp3` como un *Kaggle Dataset* (o dentro del mismo dataset del proyecto) y añádelo con **Add Data**.
2. Ajusta la variable `custom_audio_path` de abajo con la ruta resultante bajo `/kaggle/input/...`.

In [ ]:
import opensmile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

if 'model_attention' not in locals() or 'le' not in locals():
    raise NameError("El modelo (model_attention) o el LabelEncoder (le) no están cargados en memoria. Ejecuta primero el Paso 6.")

# Ajusta esta ruta al audio propio que hayas añadido como Input de Kaggle
custom_audio_path = '/kaggle/input/mi-audio-de-prueba/cancion.mp3'

if not os.path.exists(custom_audio_path):
    print(f"⚠️ No se encontró el archivo: {custom_audio_path}")
    print("Añade tu audio como Input (Add Data) y actualiza 'custom_audio_path' antes de re-ejecutar esta celda.")
else:
    print(f"\nProcesando el archivo: {custom_audio_path}...")

    smile = opensmile.Smile(
        feature_set=opensmile.FeatureSet.ComParE_2016,
        feature_level=opensmile.FeatureLevel.LowLevelDescriptors,
    )

    try:
        print("Extrayendo características acústicas (esto puede tardar unos segundos)...")
        features = smile.process_file(custom_audio_path)

        seq = features.select_dtypes(include=[np.number]).values
        print(f"Frames extraídos: {seq.shape[0]} (características por frame: {seq.shape[1]})")

        if seq.shape[1] < num_features_per_frame:
            pad_feat = num_features_per_frame - seq.shape[1]
            seq = np.pad(seq, ((0, 0), (0, pad_feat)), mode='constant')
        elif seq.shape[1] > num_features_per_frame:
            seq = seq[:, :num_features_per_frame]

        target_len = max_seq_len
        if seq.shape[0] < target_len:
            pad_width = target_len - seq.shape[0]
            seq_padded = np.pad(seq, ((0, pad_width), (0, 0)), mode='constant')
            print(f"Aplicado padding de {pad_width} frames para alcanzar la longitud esperada.")
        else:
            seq_padded = seq[:target_len, :]
            print(f"Audio truncado a los primeros {target_len} frames.")

        X_infer = np.expand_dims(seq_padded, axis=0)

        print("\n🔮 Calculando la emoción...")
        pred_probs = model_attention.predict(X_infer, verbose=0)[0]

        print("\n--- Resultados de la Predicción ---")
        for i, clase in enumerate(le.classes_):
            print(f"{clase}: {pred_probs[i]*100:.2f}%")

        clase_predominante = le.classes_[np.argmax(pred_probs)]
        print(f"\n>>> La emoción predominante es: **{clase_predominante.upper()}** <<<")

        plt.figure(figsize=(8, 4))
        sns.barplot(x=pred_probs, y=le.classes_, palette='viridis')
        plt.title(f'Distribución de Probabilidades: {os.path.basename(custom_audio_path)}')
        plt.xlabel('Probabilidad')
        plt.xlim(0, 1)
        plt.grid(axis='x', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"\n❌ Error al procesar el audio: {e}")

## Paso 8: Guardar el Modelo

Guardamos el modelo en formato `.keras` y el `LabelEncoder` con `pickle` dentro de `/kaggle/working`. Al finalizar la ejecución, usa **Save Version → Save & Run All (Commit)** para que estos archivos queden disponibles como *Output* del notebook y puedas descargarlos.

In [ ]:
import pickle
import os

model_filename = os.path.join(work_dir, 'modelo_emocion_atencion.keras')
model_attention.save(model_filename)
print(f"Modelo guardado con éxito como: {model_filename}")

le_filename = os.path.join(work_dir, 'label_encoder.pkl')
with open(le_filename, 'wb') as f:
    pickle.dump(le, f)
print(f"LabelEncoder guardado con éxito como: {le_filename}")